In [ ]:
%%html
<style>
    h1 {color:darkblue}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * **02-05. Tools**
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. **Image Generation and Editing with `ImageGenerationTool`**
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-06. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---
# Image Generation and Editing with `ImageGenerationTool`
* OpenAI hosted tool 
* Agent decides whether to call the tool based on user prompt
* Edit an existing image by sending it as a base64 `data:` URL alongside the edit prompt
* If you want to perform multi-turn image updates, maintain conversation state as we did in earlier examples (e.g., `previous_response_id`)


---

## Imports and Constants

In [ ]:
import base64
from pathlib import Path

from IPython.display import Image, Markdown, display
import source.util as util
from agents import Agent, ImageGenerationTool, Runner, trace

IMAGE_MODEL = 'gpt-image-2'
OUTPUT_DIR  = Path('resources') / 'outputs'

---
## `save_image` Utility Function
* Extracts base64-encoded result from first `image_generation_call` item in the completed run and writes it to disk


In [ ]:
def save_image(result, path: Path) -> Path:
    """Write the first generated image from a completed run to disk."""

    # next() takes first image_generation_call result from the generator expression
    image_call = next(item.raw_item for item in result.new_items
        if getattr(item.raw_item, 'type', None) == 'image_generation_call')

    # create folder structure to save image if it does not exist
    path.parent.mkdir(parents=True, exist_ok=True) 

    # decode Base64 image data and write image bytes to path
    path.write_bytes(base64.b64decode(image_call.result))
    return path

---
## Image Agent

* Optional: Configure `ImageGenerationTool` 
    * We specified image model, size, and quality
* `size='auto'` lets the model choose orientation based on the prompt or source image
* The agent instructions tell the model when and how to use the tool


In [ ]:
image_agent = Agent(
    name='ImageGenerationAssistant',
    model='gpt-5.5',
    instructions="""You are an image-generation assistant.
        Use the image_generation tool when the user asks you to
        generate, restyle, refine, or transform an image. Produce one
        high-quality PNG image per request. Choose an appropriate 
        orientation based on the prompt or source image.""",
    tools=[
        ImageGenerationTool(
            tool_config={
                'type': 'image_generation',
                'model': IMAGE_MODEL,
                'size': 'auto',
                'quality': 'high'
            }
        )
    ]
)

---

## Demo 1: Generate an Image
* Agent knows from its instructions that it's an image-generation assistant
* Prompt only describes a scene without explicitly saying to create an image
* Agent's model uses its instructions and the prompt to determine that `ImageGenerationTool` is required

In [ ]:
prompt = """A cinematic robot tutoring Python in a 
futuristic classroom, warm lighting."""

with trace('02-05-02-image-generate'):
    result1 = await Runner.run(image_agent, prompt)

path1 = save_image(result1, OUTPUT_DIR / 'image_demo1.png')
display(Markdown(result1.final_output))
display(Image(filename=str(path1), width=400))

---

## Demo 2: Image Edit for a Local Image Via a Prompt
* Represent local image as a base64 data URL
* Bundled base64 URL with the edit prompt in a multipart input
* `util.create_data_url` reads the image and encodes it
* Source image 
    > <img src="./resources/sunset.jpg" width=400 />

In [ ]:
source_image = Path('resources') / 'sunset.jpg'
edit_prompt  = """Restyle this image as Japanese anime with neon colors
and add a flying dragon."""

edit_input = [
    {
        'role': 'user',
        'content': [
            {
                'type': 'input_text',  
                'text': edit_prompt
            },
            {
                'type': 'input_image', 
                'image_url': util.create_data_url(source_image)
            }
        ]
    }
]

with trace('02-05-02-image-restyle-and-edit'):
    result2 = await Runner.run(image_agent, edit_input)

path2 = save_image(result2, OUTPUT_DIR / 'anime_sunset.png')
display(Markdown(result2.final_output))
display(Image(filename=str(path2), width=400))

---

## Defensive Coding for These Examples

* Validate style transfer input structure before sending — missing `input_image` silently produces a generation, not an edit
* Confirm local image paths exist before calling `create_data_url`
* Handle the case where no `image_generation_call` item is returned
    * For example, the request might violate OpenAI content policy
* Avoid overwriting files — generate unique filenames per run
* Catch API errors, rate limits, timeouts, and content-moderation failures
* Support explicit output formats, compression, and dimensions when downstream tools require them

---

## Documentation References

* [OpenAI Responses API image-generation tool](https://platform.openai.com/docs/guides/tools-image-generation)
* [OpenAI Agents SDK: Tools](https://openai.github.io/openai-agents-python/tools/)

---

© 2026 by Deitel & Associates, Inc. All Rights Reserved.